# 🏆 FIFA World Cup 2026 – AI Match Prediction Model
### YouTube Tutorial Notebook

**What we build in this notebook:**
1. Load 130+ years of international match data (free, no API key)
2. Build a dynamic Elo rating system to rank every national team
3. Engineer features: form, head-to-head, tournament context
4. Train an XGBoost classifier to predict Home Win / Draw / Away Win
5. Generate expected goals (xG) estimates
6. Run a Monte Carlo tournament simulation
7. Export YouTube-ready charts

---


## 0. Setup

In [ ]:
# Run once in terminal: pip install -r requirements.txt
import sys
sys.path.insert(0, '..')  # if running from a subdirectory

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

# Import our custom modules
from predictor import (
    load_results, EloRatingSystem, build_features,
    train_model, predict_match, print_prediction,
    plot_match_prediction, plot_elo_rankings,
    simulate_tournament, plot_championship_probs,
    WC2026_MATCHES, COLORS, VIS_DIR
)

print('✅ All imports OK')

## 1. Load Data (free, no API key required)

In [ ]:
df = load_results()
print(f'\n📅 Date range: {df["date"].min().date()} → {df["date"].max().date()}')
df.tail(5)

## 2. Build Elo Ratings

In [ ]:
elo = EloRatingSystem().fit(df)
top20 = elo.top_n(20)
print('\n🏅 Top 20 teams by Elo rating:')
print(top20.to_string(index=False))

In [ ]:
# Save the rankings chart for YouTube
plot_elo_rankings(elo, save_path=VIS_DIR / 'elo_rankings.png', top_n=20)

## 3. Feature Engineering

In [ ]:
feat_df = build_features(df, elo)
feat_df[['home_team','away_team','elo_diff','form_diff','h2h_home_winrate','outcome']].tail(8)

## 4. Train the Model

In [ ]:
model = train_model(feat_df)

In [ ]:
# Feature importance – great for a 'how does it work' YouTube segment
import xgboost as xgb
from predictor import FEATURE_COLS

fig, ax = plt.subplots(figsize=(10, 5), facecolor=COLORS['bg'])
ax.set_facecolor(COLORS['bg'])

importances = model.feature_importances_
idx = np.argsort(importances)

ax.barh([FEATURE_COLS[i] for i in idx], importances[idx],
        color=COLORS['home'], edgecolor='none')
ax.tick_params(colors=COLORS['text'])
ax.spines[:].set_visible(False)
ax.set_title('Feature Importance — What drives the prediction?',
             color=COLORS['accent'], fontsize=13, fontweight='bold', pad=10)

plt.tight_layout()
plt.savefig(VIS_DIR / 'feature_importance.png', dpi=150,
            bbox_inches='tight', facecolor=COLORS['bg'])
plt.show()

## 5. Predict Group-Stage Matches

In [ ]:
# ⭐ Change these to any upcoming WC26 match!
HOME_TEAM = 'Brazil'
AWAY_TEAM = 'Germany'

pred = predict_match(HOME_TEAM, AWAY_TEAM, elo, model, feat_df)
print_prediction(pred)

# Save chart → use in video thumbnail
plot_match_prediction(
    pred,
    save_path=VIS_DIR / f'pred_{HOME_TEAM.lower()}_{AWAY_TEAM.lower()}.png'
)

In [ ]:
# Predict ALL WC26 group stage matches (batch mode)
results = []
for home, away, date in WC2026_MATCHES:
    p = predict_match(home, away, elo, model, feat_df)
    p['date'] = date
    results.append(p)

predictions_df = pd.DataFrame(results)[[
    'date','home','away','p_home','p_draw','p_away','xg_home','xg_away','favorite'
]]
predictions_df.to_csv(VIS_DIR / 'group_stage_predictions.csv', index=False)
print('\n💾 Predictions saved!')
predictions_df

## 6. Tournament Simulation

In [ ]:
# Monte Carlo: 10,000 tournaments
sim_results = simulate_tournament(elo, model, feat_df, n_sims=10_000)
print('\n🏆 Championship probabilities:')
print(sim_results.head(12).to_string(index=False))

plot_championship_probs(sim_results, save_path=VIS_DIR / 'championship_probs.png')

## 7. Next Steps for Your YouTube Channel

| Video idea | What to show | Code section |
|---|---|---|
| Pre-match preview | `plot_match_prediction()` chart | Section 5 |
| Power rankings | `plot_elo_rankings()` chart | Section 2 |
| 'Who wins WC26?' | `plot_championship_probs()` | Section 6 |
| Tutorial: build xG model | Walk through feature_engineering | Section 3 |
| Post-match: was AI right? | Compare pred vs actual score | Section 5 |

**Live data upgrade:** When the tournament starts, replace `load_results()` with
live data from the free [BALLDONTLIE FIFA API](https://fifa.balldontlie.io/) or
[openfootball JSON](https://github.com/openfootball/worldcup.json).
